In [ ]:
# ============================================================
# CONFIGURATION - every path comes from config/paths.py, the single
# source of truth. Override cluster locations with the MUSICA_ENV_*
# environment variables documented there. Do not hard-code paths here.
# ============================================================
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
_ROOT = next(p for p in [_here, *_here.parents]
             if (p / 'config' / 'paths.py').exists())
sys.path.insert(0, str(_ROOT))
import config  # also puts functions/ on sys.path
from config import paths as P


In [ ]:
# To make sure there is no NAN values

In [ ]:
### Read in original emissions data
OriginalCAMS_diri = f'{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4/'

Edit_diri = f'{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/nonNANne30np4/'

In [ ]:
# Search for files that match the pattern

import glob
import os

# Directory where files are stored
# CAMS_diri = '/path/to/your/files'  # Make sure this is correctly defined

# Updated pattern with wildcard
pattern = os.path.join(OriginalCAMS_diri, 'CAMS-GLOB-ANT_ne30np4_*_v6.2_monthly.nc')

# Get list of matching files in full path
CAMS_file_list = glob.glob(pattern)
# print(CAMS_v51_file_list)

# Extract only the filenames
CAMS_file_names = [os.path.basename(f) for f in CAMS_file_list]

print(CAMS_file_names[:3])

spc_ls = []
for spcIdx in range(len(CAMS_file_names)):
    spc_name = CAMS_file_names[spcIdx].split('_')[4]
    spc_ls.append(spc_name)
    
import re
# Extract species name between 'ne30np4_' and '_c20210423'
species_names = [
    re.search(r'ne30np4_(.+?)_v6.2_monthly', f).group(1)
    for f in CAMS_file_names
]
species_names = sorted(species_names)
print(species_names)    

In [ ]:
import os
import xarray as xr
import numpy as np

# Directories
OriginalCAMS_diri = f'{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4/'
Edit_diri = f'{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/nonNANne30np4/'

# List of target species
targetspc_ls = species_names
# targetspc_ls = ['CO', 'NO', 'CH2O', 'C2H6']  # update as needed

# Coordinate-like vars to clean attributes (and fill NaN only if numeric)
coord_like_vars = {'time', 'ncol', 'lat', 'lon', 'area', 'date', 'altitude'}

# Ensure output directory exists
os.makedirs(Edit_diri, exist_ok=True)

for spc in targetspc_ls:
    spc_fileINpath = os.path.join(OriginalCAMS_diri, f'CAMS-GLOB-ANT_ne30np4_{spc}_v6.2_monthly.nc')
    spc_fileOUTpath = os.path.join(Edit_diri, f'CAMS-GLOB-ANT_ne30np4_{spc}_v6.2_monthly.nc')

    print(f'Processing {spc_fileINpath}...')

    # Load dataset
    ds = xr.open_dataset(spc_fileINpath)
            
    # Clean non-coordinate variables
    for var in ds.data_vars:
        if var not in coord_like_vars:
            if np.issubdtype(ds[var].dtype, np.number):
                ds[var] = ds[var].fillna(0)
            ds[var].attrs.pop('_FillValue', None)
            ds[var].encoding.pop('_FillValue', None)  # <<<< THIS IS CRUCIAL
            ds[var].encoding['_FillValue'] = None

    # Clean coordinate-like variables
    for var in coord_like_vars:
        if var in ds:
            if np.issubdtype(ds[var].dtype, np.number):
                ds[var] = ds[var].fillna(0)
            ds[var].attrs.pop('_FillValue', None)
            ds[var].encoding.pop('_FillValue', None)  # <<<< ALSO HERE
            ds[var].encoding['_FillValue'] = None


    # Save cleaned file
    ds.to_netcdf(spc_fileOUTpath)
    print(f'Saved cleaned file to {spc_fileOUTpath}')
